# P7 — MHSDF (CNN + BiLSTM) Pipeline

**Local execution notebook** for all P7 variations (A / B / C / D) across all 3 text modes.

## What this notebook does
1. Verifies the environment and dataset paths
2. Builds the BERT tokenizer
3. Runs a smoke test (1-epoch, 200-sample subset) to catch bugs fast
4. Trains all selected P7 variations
5. Evaluates on the test set and prints a comparison table

## Quick-start
- **Just P7-D (recommended first run):** Set `RUN_VARIATIONS = ['D']` and `TEXT_MODES = ['tweet_ocr']` in Cell 2.
- **Full ablation (12 runs):** Set `RUN_VARIATIONS = ['A','B','C','D']` and `TEXT_MODES = ['no_caption','tweet_ocr','all_text']`.
- **Smoke test only:** Set `SMOKE_TEST_ONLY = True`.

## Structure of `src/p7/`
```
src/p7/
  __init__.py    — package init
  config.py      — P7Config dataclass
  tokenizer.py   — BERT tokenizer wrapper
  dataset.py     — P7Dataset (stage filter, text modes, multi-label)
  model.py       — MHSDF (CNNVisualEncoder + BiLSTMTextEncoder)
  losses.py      — Loss factory
  trainer.py     — Two-stage training orchestrator
  metrics.py     — Multi-label metrics + threshold calibration
```

In [1]:
# Cell 1: Environment setup
import sys, os
from pathlib import Path

# Add project root to path (works whether you run from notebooks/ or root)
PROJECT_ROOT = Path().resolve().parent if Path().resolve().name == 'notebooks' else Path().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')
print(f'Python: {sys.version}')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Project root: C:\Users\ekans\Desktop\Btech\Sem-4\DL\EndSem_Project
Python: 3.13.13 (tags/v3.13.13:01104ce, Apr  7 2026, 19:25:48) [MSC v.1944 64 bit (AMD64)]
PyTorch: 2.6.0+cu124
CUDA available: True
GPU: NVIDIA GeForce GTX 1650


In [2]:
# Cell 2: Configuration — EDIT THIS to control what runs

# Which variations to run
RUN_VARIATIONS = ['D']           # Options: 'A', 'B', 'C', 'D'

# Which text modes to run
TEXT_MODES = ['tweet_ocr']       # Options: 'no_caption', 'tweet_ocr', 'all_text'

# Run only a 1-epoch smoke test on a 200-sample subset before full training?
SMOKE_TEST_ONLY = False

# Override training epochs (set None to use defaults from P7Config)
S1_EPOCHS_OVERRIDE = None        # e.g. 3 for quick test
S2_EPOCHS_OVERRIDE = None

# Device (leave 'auto' to auto-detect CUDA)
DEVICE = 'auto'

# DataLoader workers (0 = single process, safe on Windows)
NUM_WORKERS = 4

# Seed
SEED = 42

MAX_TRAIN_SAMPLES = 30_000    # None = full dataset
MAX_VAL_SAMPLES   = None      # keep full val for reliable metrics


print(f'Will run: variations={RUN_VARIATIONS}  text_modes={TEXT_MODES}')
print(f'Total runs: {len(RUN_VARIATIONS) * len(TEXT_MODES)}')

Will run: variations=['D']  text_modes=['tweet_ocr']
Total runs: 1


In [3]:
# Cell 3: Verify dataset paths
from src.utils.config import (
    PROJECT_ROOT, GT_FILE, IMG_DIR, PROCESSED_LABELS_FILE,
    SPLITS_DIR, RESULTS_DIR
)

CAPTIONS_FILE = PROJECT_ROOT / 'results' / 'vlm_captions.json'

checks = [
    ('GT JSON',            GT_FILE),
    ('Images dir',         IMG_DIR),
    ('Processed labels',   PROCESSED_LABELS_FILE),
    ('Train split IDs',    SPLITS_DIR / 'train_ids.txt'),
    ('Val split IDs',      SPLITS_DIR / 'val_ids.txt'),
    ('Test split IDs',     SPLITS_DIR / 'test_ids.txt'),
    ('VLM captions',       CAPTIONS_FILE),
]

all_ok = True
for name, path in checks:
    status = '✓' if path.exists() else '✗ MISSING'
    if not path.exists() and name != 'VLM captions':
        all_ok = False
    print(f'  {status}  {name}: {path}')

if not all_ok:
    raise RuntimeError('Required files missing. Check your dataset directory.')

if not CAPTIONS_FILE.exists():
    print('\n⚠️  VLM captions not found — all_text mode will fall back to tweet+OCR.')

  ✓  GT JSON: C:\Users\ekans\Desktop\Btech\Sem-4\DL\EndSem_Project\dataset\MMHS150K_GT.json
  ✓  Images dir: C:\Users\ekans\Desktop\Btech\Sem-4\DL\EndSem_Project\dataset\img_resized
  ✓  Processed labels: C:\Users\ekans\Desktop\Btech\Sem-4\DL\EndSem_Project\dataset\processed_labels.json
  ✓  Train split IDs: C:\Users\ekans\Desktop\Btech\Sem-4\DL\EndSem_Project\dataset\splits\train_ids.txt
  ✓  Val split IDs: C:\Users\ekans\Desktop\Btech\Sem-4\DL\EndSem_Project\dataset\splits\val_ids.txt
  ✓  Test split IDs: C:\Users\ekans\Desktop\Btech\Sem-4\DL\EndSem_Project\dataset\splits\test_ids.txt
  ✓  VLM captions: C:\Users\ekans\Desktop\Btech\Sem-4\DL\EndSem_Project\results\vlm_captions.json


In [4]:
# Cell 4: Load tokenizer (downloads once from Hugging Face Hub)
import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    datefmt='%H:%M:%S',
)

from src.p7.tokenizer import P7Tokenizer

tokenizer = P7Tokenizer(
    model_name='cardiffnlp/twitter-roberta-base',
    max_seq_len=128,
)
print(tokenizer)

# Quick encode test
test_ids = tokenizer.encode('This is a racist meme. KKK')
print(f'Test encode: {len(test_ids)} tokens  (first 10: {test_ids[:10]})')

14:47:19 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/cardiffnlp/twitter-roberta-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
14:47:19 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cardiffnlp/twitter-roberta-base/cbb417e9647b51504caf68cbe1af6bbf56da06b7/config.json "HTTP/1.1 200 OK"
14:47:19 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/cardiffnlp/twitter-roberta-base/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
14:47:19 [WARNING] huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
14:47:19 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/cardiffnlp/twitter-roberta-base/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
14:47:20 [INFO] httpx: HTTP Request: GET https://huggingface.co/api/models/cardiffnlp/twitter-roberta-base/tree/main/additional_chat_templates?recursiv

P7Tokenizer(model='cardiffnlp/twitter-roberta-base', vocab_size=50265, max_seq_len=128)
Test encode: 128 tokens  (first 10: [0, 713, 16, 10, 7159, 25426, 4, 34829, 2, 1])


In [5]:
# Cell 5: Dataset smoke test — verify loading & multi-label labels
from src.p7.config import P7Config
from src.p7.dataset import P7Dataset, HATE_CAT_NAMES, compute_multilabel_from_soft

cfg_test = P7Config(variation='D', text_mode='tweet_ocr')

# Load a tiny subset to verify the dataset works
val_ds = P7Dataset('val', cfg_test, tokenizer, stage=None, is_training=False)

sample = val_ds[0]
print('Sample keys:', list(sample.keys()))
print(f'  image shape:           {sample["image"].shape}')
print(f'  token_ids shape:       {sample["token_ids"].shape}')
print(f'  text (first 80 chars): {sample["text"][:80]}')
print(f'  label_binary:          {sample["label_binary"]}')
print(f'  label_6class:          {sample["label_6class"]}')
print(f'  label_s2:              {sample["label_s2"]}  (-1 = NotHate)')
print(f'  multi_label_binary:    {sample["multi_label_binary"].tolist()}')
print(f'  soft_label:            {[round(x,3) for x in sample["soft_label"].tolist()]}')

# Verify Stage 2 filtering
val_s2 = P7Dataset('val', cfg_test, tokenizer, stage=2, is_training=False)
labels_s2 = [val_s2.labels[sid]['hard_label_binary'] for sid in val_s2.sample_ids]
assert all(l == 1 for l in labels_s2), 'Stage 2 filter bug: NotHate samples present!'
print(f'\nStage 2 filter check: {len(val_s2)} hateful-only samples  ✓')

14:47:23 [INFO] src.p7.dataset: [P7] Loaded 79,256 VLM captions.
14:47:23 [INFO] src.p7.dataset: [P7] val dataset: 5,000 samples  (stage=None)
14:47:23 [INFO] src.data.image_store: [ImageStore] Loaded: 150,000 images  (22.6GB on disk)  img_size=224
14:47:23 [INFO] src.p7.dataset: [P7] val Stage 2: Kept 1733/5000 hateful-only samples.
14:47:23 [INFO] src.p7.dataset: [P7] val dataset: 1,733 samples  (stage=2)


Sample keys: ['image', 'token_ids', 'text', 'tweet_id', 'label_binary', 'label_6class', 'label_s2', 'multi_label_binary', 'soft_label', 'agreement_level']
  image shape:           torch.Size([3, 224, 224])
  token_ids shape:       torch.Size([128])
  text (first 80 chars): fuck my pussy chaturbate cut fingering happy tugs hot cunt tyra misoux switzerla
  label_binary:          1
  label_6class:          2
  label_s2:              1  (-1 = NotHate)
  multi_label_binary:    [0.0, 0.0, 1.0, 0.0, 0.0, 0.0]
  soft_label:            [0.333, 0.0, 0.667, 0.0, 0.0, 0.0]

Stage 2 filter check: 1733 hateful-only samples  ✓


In [6]:
# Cell 6: Model smoke test — verify forward pass shapes
import torch
from src.p7.model import MHSDF
from src.p7.config import P7Config

cfg = P7Config(variation='D', text_mode='tweet_ocr')
vocab_size = tokenizer.vocab_size

for nc, label in [(2, 'Stage 1 binary'), (5, 'Stage 2 hate cats'), (6, 'P7-B 6-class')]:
    m = MHSDF.from_config(cfg, vocab_size, num_classes=nc)
    imgs = torch.randn(4, 3, 224, 224)
    toks = torch.randint(0, vocab_size, (4, 128))
    out = m(imgs, toks)
    assert out.shape == (4, nc), f'Expected (4,{nc}), got {out.shape}'
    n_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'  num_classes={nc} ({label}): logits {out.shape}  params={n_params:,}')

print('\nForward pass check: all shapes correct ✓')

  num_classes=2 (Stage 1 binary): logits torch.Size([4, 2])  params=11,795,458
  num_classes=5 (Stage 2 hate cats): logits torch.Size([4, 5])  params=11,796,229
  num_classes=6 (P7-B 6-class): logits torch.Size([4, 6])  params=11,796,486

Forward pass check: all shapes correct ✓


In [7]:
# Cell 7: Smoke test — 1 epoch on 200-sample subset
# Catches integration bugs before committing to full training.

import torch
from torch.utils.data import Subset
from torch.utils.data import DataLoader
from src.p7.dataset import p7_collate_fn
from src.p7.model import MHSDF
from src.p7.losses import get_p7_loss
from src.p7.trainer import resolve_device

device = resolve_device(DEVICE)
cfg_smoke = P7Config(
    variation='D', text_mode='tweet_ocr',
    s1_epochs=1, s2_epochs=1, device=DEVICE,
    num_workers=NUM_WORKERS, seed=SEED,
)

train_full = P7Dataset('train', cfg_smoke, tokenizer, stage=1, is_training=True)
smoke_ds   = Subset(train_full, list(range(min(200, len(train_full)))))

class SubsetWrapper(torch.utils.data.Dataset):
    """Thin wrapper so loss factory can access .labels and .sample_ids."""
    def __init__(self, subset):
        self._ds = subset
        self.labels = subset.dataset.labels
        self.sample_ids = [subset.dataset.sample_ids[i] for i in subset.indices]
        self.config = subset.dataset.config
    def __len__(self): return len(self._ds)
    def __getitem__(self, i): return self._ds[i]

smoke_wrapped = SubsetWrapper(smoke_ds)
smoke_loader = DataLoader(smoke_wrapped, batch_size=16, collate_fn=p7_collate_fn, shuffle=True)

model = MHSDF.from_config(cfg_smoke, tokenizer.vocab_size, num_classes=1).to(device)
criterion = get_p7_loss('D', 1, smoke_wrapped, device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

model.train()
losses = []
for batch in smoke_loader:
    imgs = batch['image'].to(device)
    toks = batch['token_ids'].to(device)
    targets = batch['label_binary'].float().to(device)
    logits = model(imgs, toks).squeeze(-1)
    loss = criterion(logits, targets)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    losses.append(loss.item())

print(f'Smoke test passed ✓  avg_loss={sum(losses)/len(losses):.4f}  n_batches={len(losses)}')

14:47:24 [INFO] src.p7.dataset: [P7] train dataset: 134,823 samples  (stage=1)
14:47:25 [INFO] src.p7.losses: [P7 Loss] Binary pos_weight: 0.399  (neg=57, pos=143)


Smoke test passed ✓  avg_loss=0.4288  n_batches=13


In [8]:
# Cell 8: MAIN TRAINING — runs selected variations and text modes
# Skip if SMOKE_TEST_ONLY = True

if SMOKE_TEST_ONLY:
    print('SMOKE_TEST_ONLY=True — skipping full training.')
else:
    import logging
    from src.p7.config import P7Config
    from src.p7.trainer import run_p7

    all_results = {}

    for variation in RUN_VARIATIONS:
        for text_mode in TEXT_MODES:
            run_name = f'p7_{variation}_{text_mode}'
            print(f'\n{"="*60}')
            print(f'  Starting: {run_name}')
            print(f'{"="*60}')

            cfg = P7Config(
                variation=variation,
                text_mode=text_mode,
                run_name=run_name,
                device=DEVICE,
                num_workers=NUM_WORKERS,
                seed=SEED,
                s1_epochs=S1_EPOCHS_OVERRIDE or (15 if variation in ('A','C','D') else 15),
                s2_epochs=S2_EPOCHS_OVERRIDE or 20,
                max_train_samples=MAX_TRAIN_SAMPLES,
                max_val_samples=MAX_VAL_SAMPLES,
            )

            metrics = run_p7(cfg)
            all_results[run_name] = metrics
            print(f'\n  ✓ Done: {run_name}')

    print(f'\n✓ All {len(all_results)} runs complete!')

14:47:29 [INFO] src.p7.trainer: [P7] Starting: p7_D_tweet_ocr  device=cuda
14:47:29 [INFO] src.data.image_store: [ImageStore] Already built at C:\Users\ekans\Desktop\Btech\Sem-4\DL\EndSem_Project\dataset\image_store_224.bin. Loading ...
14:47:29 [INFO] src.p7.dataset: [P7] Loading token cache from C:\Users\ekans\Desktop\Btech\Sem-4\DL\EndSem_Project\dataset\token_cache_tweet_ocr_128.npy ...



  Starting: p7_D_tweet_ocr


14:47:29 [INFO] src.p7.dataset: [P7] Token cache loaded: 149,823 samples.
14:47:29 [INFO] src.p7.dataset: [P7] train: Stratified subset → 30,000 samples (max_samples=30000)
14:47:29 [INFO] src.p7.dataset: [P7] train dataset: 30,000 samples  (stage=1)
14:47:29 [INFO] src.p7.dataset: [P7] val dataset: 5,000 samples  (stage=1)
14:47:29 [INFO] src.p7.losses: [P7 Loss] Binary pos_weight: 5.542  (neg=25414, pos=4586)
C:\Users\ekans\Desktop\Btech\Sem-4\DL\EndSem_Project\src\p7\trainer.py:248: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  torch.cuda.amp.GradScaler()
14:47:29 [INFO] src.p7.trainer: [P7 Trainer] AMP (FP16) enabled.
14:47:29 [INFO] src.p7.trainer: 
[P7 Trainer] variation=D  stage=1  epochs=15  lr=0.001  batch=64  device=cuda


KeyboardInterrupt: 

In [ ]:
# Cell 9: Results comparison table
# Reads metrics.json for each completed run and prints a summary.

from pathlib import Path
import json

results_base = PROJECT_ROOT / 'results' / 'p7'

rows = []
for run_dir in sorted(results_base.iterdir()):
    mfile = run_dir / 'metrics.json'
    if not mfile.exists():
        continue
    with open(mfile) as f:
        m = json.load(f)

    variation = m.get('variation', '?')
    run_name  = m.get('config', run_dir.name)

    # Extract key metrics
    s1_f1 = m.get('stage1', {}).get('best', None)
    s2_key = 'stage2_calibrated' if 'stage2_calibrated' in m else 'stage2'
    s2_f1 = m.get(s2_key, {}).get('best', None) or m.get(s2_key, {}).get('multilabel/macro_f1', None)
    composite = m.get('pipeline', {}).get('pipeline/composite', None)

    rows.append({
        'run':       run_name,
        'var':       variation,
        's1_f1':     f'{s1_f1:.4f}' if s1_f1 else '—',
        's2_f1':     f'{s2_f1:.4f}' if s2_f1 else '—',
        'composite': f'{composite:.4f}' if composite else '—',
    })

# Print table
if rows:
    header = f'{"Run":<30} {"Var":<5} {"S1 Macro F1":<14} {"S2 Macro F1":<14} {"Composite":<12}'
    print(header)
    print('-' * len(header))
    for r in rows:
        print(f'{r["run"]:<30} {r["var"]:<5} {r["s1_f1"]:<14} {r["s2_f1"]:<14} {r["composite"]:<12}')
else:
    print('No completed runs found in results/p7/')

In [ ]:
# Cell 10: Test-set evaluation
# Runs the best saved model on the test split.

if SMOKE_TEST_ONLY:
    print('Skipping test eval in smoke-test mode.')
else:
    import torch
    import numpy as np
    from torch.utils.data import DataLoader
    from src.p7.config import P7Config
    from src.p7.dataset import P7Dataset, p7_collate_fn
    from src.p7.model import MHSDF
    from src.p7.metrics import apply_thresholds, compute_multilabel_metrics
    from src.p7.trainer import resolve_device
    from src.evaluation.metrics import compute_binary_metrics

    # Evaluate the primary run (P7-D tweet_ocr) on test
    PRIMARY_RUN = 'p7_D_tweet_ocr'
    ckpt_dir  = PROJECT_ROOT / 'checkpoints' / 'p7' / PRIMARY_RUN
    results_dir = PROJECT_ROOT / 'results' / 'p7' / PRIMARY_RUN

    cfg = P7Config.load(results_dir / 'config.yaml')
    device = resolve_device(cfg.device)

    from src.p7.tokenizer import P7Tokenizer
    tok = P7Tokenizer(cfg.bert_model_name, cfg.max_seq_len)

    # Stage 1 test eval
    test_s1 = P7Dataset('test', cfg, tok, stage=1, is_training=False)
    s1_loader = DataLoader(test_s1, batch_size=128, collate_fn=p7_collate_fn)

    model_s1 = MHSDF.from_config(cfg, tok.vocab_size, num_classes=1).to(device)
    model_s1.load_state_dict(torch.load(ckpt_dir / 'stage1_best.pt', map_location=device))
    model_s1.eval()

    all_logits_s1, all_targets_s1 = [], []
    with torch.no_grad():
        for batch in s1_loader:
            logits = model_s1(batch['image'].to(device), batch['token_ids'].to(device))
            all_logits_s1.append(logits.cpu())
            all_targets_s1.append(batch['label_binary'])

    logits_s1 = torch.cat(all_logits_s1).squeeze(-1).numpy()
    targets_s1 = torch.cat(all_targets_s1).numpy()
    probs_s1 = 1 / (1 + np.exp(-logits_s1))
    preds_s1 = (probs_s1 >= 0.5).astype(int)

    s1_metrics = compute_binary_metrics(targets_s1, preds_s1, probs_s1)
    print('=== Stage 1 (Binary) — TEST ===')
    for k, v in s1_metrics.items():
        print(f'  {k}: {v:.4f}')

    # Stage 2 test eval (P7-D multilabel)
    test_s2 = P7Dataset('test', cfg, tok, stage=2, is_training=False)
    s2_loader = DataLoader(test_s2, batch_size=128, collate_fn=p7_collate_fn)

    model_s2 = MHSDF.from_config(cfg, tok.vocab_size, num_classes=5).to(device)
    model_s2.load_state_dict(torch.load(ckpt_dir / 'stage2_best.pt', map_location=device))
    model_s2.eval()

    thresholds = np.load(ckpt_dir / 'stage2_thresholds.npy')

    all_logits_s2, all_targets_s2 = [], []
    with torch.no_grad():
        for batch in s2_loader:
            logits = model_s2(batch['image'].to(device), batch['token_ids'].to(device))
            all_logits_s2.append(logits.cpu())
            all_targets_s2.append(batch['multi_label_binary'][:, 1:])  # hate cats only

    logits_s2 = torch.cat(all_logits_s2).numpy()
    targets_s2 = torch.cat(all_targets_s2).numpy()
    preds_s2 = apply_thresholds(logits_s2, thresholds)

    from src.p7.metrics import compute_multilabel_metrics, HATE_CAT_NAMES
    s2_metrics = compute_multilabel_metrics(targets_s2.astype(int), preds_s2, HATE_CAT_NAMES)
    print('\n=== Stage 2 (Multilabel) — TEST ===')
    for k, v in s2_metrics.items():
        if isinstance(v, float):
            print(f'  {k}: {v:.4f}')